# CS781 Phase 3 — Rhetorical Role Prediction (CL + IT): training pipeline

**What this notebook does:** loads the Phase 3 data, builds sentence-only and context-aware inputs, fine-tunes your Phase 1 language models with
six adaptation techniques (full FT, LoRA/rsLoRA, **LoRMA+/LoRMAπ**, IA³, QLoRA-style NF4, prompt tuning) plus **knowledge distillation**,
logs everything (macro-F1/P/R, trainable params, model size, GPU memory, time) to `ledger.csv`, and writes `prediction.json` + `submission.zip`.

**Run order:** 1 setup → 2 checkpoint inventory → 3 data → 4 tokenizer/examples → 5 stats → 6 metrics → 7 models → 8 trainer → **9 smoke tests (must pass)** → 10 plan → 11 RUN A → 12 RUN B → 13 KD → 14 summary → 15 post-hoc → 16 submission.

**Read before running**
* The dataset is gated, so cell 3 auto-detects the JSON layout and prints it. If it fails, paste the printed schema and set `CUSTOM_PARSER`.
* Only the GPT-2-style Phase 1 checkpoint has a loader. Cell 2 lists every checkpoint with a family guess; the other three architectures need a loader in cell 7 (send the cell 2 output).
* Check the output format of cell 16 against the official `prediction.json` schema (`COMBINE_MODE`).
* Dev has only a few documents per domain, so differences below ~0.02 macro-F1 are noise. Trust results that agree across CL and IT.


In [ ]:
# ===== CELL 6 — REGISTER THE CORRECT PHASE 1 CHECKPOINT =====

import os

PHASE1_CHECKPOINT = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase-1-checkpoints/checkpoint_latest.pt"
)

assert os.path.isfile(PHASE1_CHECKPOINT), (
    f"Checkpoint not found: {PHASE1_CHECKPOINT}. "
    "Make sure CS781 Phase 1 Checkpoints is attached to Kaggle."
)

ARCHS = {
    "gpt2_a": {
        "path": PHASE1_CHECKPOINT,
        "family": "gpt2",
    }
}

print("Registered architecture:", list(ARCHS.keys()))
print("Checkpoint:", ARCHS["gpt2_a"]["path"])

In [ ]:
# ================= CELL 0 — optional Kaggle secrets =================
# Do not require unrelated Kaggle API secrets just to run the notebook.
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    try:
        HF_TOKEN = _secrets.get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
except Exception:
    pass

print("HF_TOKEN secret available:", bool(HF_TOKEN))


In [ ]:
# ================= CELL 1 — environment, paths, helpers =================
import os, sys, json, math, time, random, re, copy, gc, glob, zipfile, shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from huggingface_hub import snapshot_download, HfApi

SEED = 42

def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device,
      torch.cuda.get_device_name(0) if device.type == "cuda"
      else "(no GPU: training will be slow)")

WORK = Path("/kaggle/working/phase3_rr")
(WORK / "runs").mkdir(parents=True, exist_ok=True)
LEDGER = WORK / "ledger.csv"

def find_files(names, roots=("/kaggle/input", "/kaggle/working", "/content", ".")):
    found = {}
    for root in roots:
        if not os.path.exists(root):
            continue
        for n in names:
            if n in found:
                continue
            hits = sorted(glob.glob(f"{root}/**/{n}", recursive=True))
            if hits:
                found[n] = hits[0]
    return found

NEED = [
    "CL_train.json", "CL_dev.json", "CL_test.json",
    "IT_train.json", "IT_dev.json", "IT_test.json",
    "label_vocab.json"
]

paths = find_files(NEED)
missing = [n for n in NEED if n not in paths]

if missing:
    print("Missing locally:", missing)
    print("Trying Hugging Face dataset download...")

    token = os.environ.get("HF_TOKEN") or HF_TOKEN

    # Kaggle Secrets fallback
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None

    if not token:
        raise RuntimeError(
            "HF_TOKEN was not found. In Kaggle, open Add-ons > Secrets, "
            "add a secret named HF_TOKEN with your Hugging Face READ token, "
            "and enable notebook access. Also accept the dataset conditions "
            "on the Hugging Face dataset page. Alternatively, attach the "
            "Phase_3_RR dataset to this Kaggle notebook."
        )

    try:
        local = snapshot_download(
            repo_id="Exploration-Lab/CS-781-Capstone",
            repo_type="dataset",
            token=token,
            allow_patterns=["Phase_3_RR/**"],
            local_dir=str(WORK / "hf_data")
        )
    except Exception as e:
        raise RuntimeError(
            "Hugging Face download failed. Check that you have accepted "
            "the dataset conditions and that HF_TOKEN is a valid token "
            "with read access. Original error: " + repr(e)
        ) from e

    paths = find_files(NEED, roots=(local, str(WORK / "hf_data")))
    missing = [n for n in NEED if n not in paths]

    if missing:
        # Fallback search if the files were placed in nested folders
        paths.update(find_files(missing, roots=(str(WORK / "hf_data"), local)))
        missing = [n for n in NEED if n not in paths]

    if missing:
        raise FileNotFoundError(
            f"Download completed, but these files were not found: {missing}. "
            "Check the dataset's folder structure."
        )

for k, v in paths.items():
    print(f"{k:18s} {v}")

print("Cell 1 setup complete.")

# Load the discovered files once. The parser cell below will convert them into DOCS.
RAW = {name: json.loads(Path(path).read_text(encoding="utf-8")) for name, path in paths.items()}
print("Loaded raw JSON files:", {k: type(v).__name__ for k, v in RAW.items()})


In [ ]:
# ================= CELL 2 — inventory of every Phase 1 checkpoint (use this to fill the architecture registry) =================
def family_guess(keys):
    s = " ".join(keys[:400])
    if "attn.c_attn" in s: return "gpt2-style (HF GPT2: fused c_attn Conv1D)"
    if "self_attn.q_proj" in s and "gate_proj" in s: return "llama-style (q_proj/k_proj/v_proj + gated MLP)"
    if "A_log" in s or "mixer" in s or "in_proj" in s: return "state-space / mamba-style"
    if "rwkv" in s.lower() or "time_mix" in s: return "rwkv-style"
    if "encoder.layer" in s or "attention.self.query" in s: return "bert-style (bidirectional encoder)"
    return "unknown - paste the key names to the assistant"

CKPT_EXT = (".pt", ".pth", ".bin", ".ckpt")
cands = sorted(p for p in glob.glob("/kaggle/input/**/*", recursive=True) if p.endswith(CKPT_EXT) and os.path.getsize(p) > 5e6)
print(f"{len(cands)} candidate checkpoint files under /kaggle/input\n")
for p in cands:
    try:
        try: obj = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
        except Exception: obj = torch.load(p, map_location="cpu", weights_only=False)
    except Exception as e:
        print(p, "-> load failed:", str(e)[:120], "\n"); continue
    sd = obj.get("model_state_dict", obj) if isinstance(obj, dict) else None
    print(p, f"({os.path.getsize(p)/2**20:.0f} MB)")
    if isinstance(obj, dict):
        print("   top-level keys:", [k for k in obj.keys()][:12])
        if "model_config" in obj: print("   model_config  :", obj["model_config"])
        for k in ("step", "epoch", "context_length"):
            if k in obj: print(f"   {k}: {obj[k]}")
    if isinstance(sd, dict) and sd and hasattr(next(iter(sd.values())), "numel"):
        keys = list(sd.keys()); n = sum(v.numel() for v in sd.values() if hasattr(v, "numel"))
        print(f"   tensors: {len(keys)} | params: {n/1e6:.1f}M | family guess: {family_guess(keys)}")
        print("   first keys:", keys[:3])
    print()
    del obj, sd; gc.collect()
print("Map each of the FOUR Phase 1 models to an entry in ARCHS (cell 7). If a family says 'unknown', send this output to get a loader.")


In [ ]:
# ================= CELL 3 — PARSE PHASE 3 DATA (TRAIN / DEV / TEST) =================
# Supports the observed test schema {doc_id: [sentence, ...]} and common labeled schemas.

LABELS_FALLBACK = [
    "Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute",
    "Dissent", "PrecedentReliedUpon", "PrecedentNotReliedUpon",
    "PrecedentOverruled", "RulingByLowerCourt", "RatioOfTheDecision",
    "RulingByPresentCourt", "None"
]

def _norm_key(k):
    return str(k).strip().lower().replace("-", "_").replace(" ", "_")

def _find_sentence_list(payload):
    if isinstance(payload, list):
        if all(isinstance(x, str) for x in payload):
            return payload
        if payload and all(isinstance(x, dict) for x in payload):
            out = []
            for item in payload:
                sent = None
                for k, v in item.items():
                    if _norm_key(k) in {"text", "sentence", "sent", "content"} and isinstance(v, str):
                        sent = v
                        break
                if sent is None:
                    out = []
                    break
                out.append(sent)
            if out:
                return out
        for item in payload:
            got = _find_sentence_list(item)
            if got is not None:
                return got
        return None

    if isinstance(payload, dict):
        # Prefer conventional keys first.
        for k, v in payload.items():
            if _norm_key(k) in {"text", "texts", "sentences", "sentence", "sents", "content"}:
                if isinstance(v, list) and all(isinstance(x, str) for x in v):
                    return v
                if isinstance(v, str):
                    return [v]
        # Search nested values, but don't treat a labels list as sentence text.
        for k, v in payload.items():
            if _norm_key(k) in {"labels", "label", "y", "tags"}:
                continue
            if isinstance(v, (dict, list)):
                got = _find_sentence_list(v)
                if got is not None:
                    return got
    return None

def _find_label_list(payload):
    if isinstance(payload, dict):
        for k, v in payload.items():
            if _norm_key(k) in {"labels", "label", "tags", "roles", "y"} and isinstance(v, list):
                return v
        for k, v in payload.items():
            if isinstance(v, (dict, list)):
                got = _find_label_list(v)
                if got is not None:
                    return got
    elif isinstance(payload, list):
        if payload and all(isinstance(x, dict) for x in payload):
            vals = []
            for item in payload:
                lab = None
                for k, v in item.items():
                    if _norm_key(k) in {"label", "labels", "tag", "role"} and isinstance(v, str):
                        lab = v
                        break
                if lab is None:
                    vals = []
                    break
                vals.append(lab)
            if vals:
                return vals
        for item in payload:
            if isinstance(item, (dict, list)):
                got = _find_label_list(item)
                if got is not None:
                    return got
    return None

def parse_split(obj, style=None):
    if isinstance(obj, dict):
        items = list(obj.items())
    elif isinstance(obj, list):
        items = [(str(i), x) for i, x in enumerate(obj)]
    else:
        raise ValueError(f"Unsupported split JSON type: {type(obj).__name__}")

    docs = []
    for did, payload in items:
        # The test files were confirmed to map document IDs directly to sentence strings.
        if isinstance(payload, list) and all(isinstance(x, str) for x in payload):
            docs.append({"id": str(did), "sents": payload, "labels": None, "setter": None})
            continue

        sents = _find_sentence_list(payload)
        labels = _find_label_list(payload)
        if sents is None:
            raise ValueError(
                f"{did}: cannot find a sentence list. Payload type={type(payload).__name__}; "
                f"keys={list(payload.keys())[:30] if isinstance(payload, dict) else 'n/a'}"
            )
        if labels is not None:
            labels = [str(x) for x in labels]
            if len(labels) != len(sents):
                raise ValueError(f"{did}: {len(sents)} sentences but {len(labels)} labels")
        docs.append({"id": str(did), "sents": [str(x) for x in sents],
                     "labels": labels, "setter": None})
    return docs, style

# Load label vocabulary if it is a recognizable format; otherwise use the official label names.
_vocab_obj = RAW.get("label_vocab.json")
LABELS = None
if isinstance(_vocab_obj, list) and all(isinstance(x, str) for x in _vocab_obj):
    LABELS = _vocab_obj
elif isinstance(_vocab_obj, dict):
    for key in ("labels", "label_vocab", "id2label", "label2id"):
        value = _vocab_obj.get(key)
        if isinstance(value, list) and all(isinstance(x, str) for x in value):
            LABELS = value
            break
        if isinstance(value, dict) and value:
            try:
                LABELS = [k for k, v in sorted(value.items(), key=lambda kv: int(kv[1]))]
            except Exception:
                LABELS = list(value.keys())
            break
if not LABELS:
    LABELS = LABELS_FALLBACK
LABEL2ID = {lab: i for i, lab in enumerate(LABELS)}
STYLE = {"CL": None, "IT": None}

DOCS = {"CL": {}, "IT": {}}
TEST_OBJ = {}
for dom in ("CL", "IT"):
    for split in ("train", "dev"):
        DOCS[dom][split], _ = parse_split(RAW[f"{dom}_{split}.json"])
        if not DOCS[dom][split]:
            raise ValueError(f"{dom} {split} split is empty")
        for d in DOCS[dom][split]:
            if d["labels"] is None:
                raise ValueError(f"{dom} {split} document {d['id']} has no labels")
            bad = sorted(set(d["labels"]) - set(LABELS))
            if bad:
                raise ValueError(f"{dom} {split} document {d['id']} has labels absent from vocabulary: {bad}")
    TEST_OBJ[dom] = RAW[f"{dom}_test.json"]
    DOCS[dom]["test"], _ = parse_split(TEST_OBJ[dom])
    print(dom, {sp: len(DOCS[dom][sp]) for sp in ("train", "dev", "test")})
    print("  sample sentence counts:", [len(d["sents"]) for d in DOCS[dom]["test"][:2]])

print("Labels:", LABELS)
print("DOCS and LABEL2ID are ready.")


In [ ]:
# ================= CELL 5 — tokenizer, example builder (sentence-only vs context-aware), batching =================
from tokenizers import Tokenizer
_tok_files = find_files(["legal_bpe_32k.json"])
if "legal_bpe_32k.json" not in _tok_files:
    raise FileNotFoundError(
        "legal_bpe_32k.json was not found. Attach the Phase 1 tokenizer dataset "
        "to Kaggle or update tok_path to its actual path."
    )
tok_path = _tok_files["legal_bpe_32k.json"]
tokenizer = Tokenizer.from_file(tok_path)
print("tokenizer:", tok_path, "| vocab:", tokenizer.get_vocab_size())

_ENC = {}
def enc(text):
    r = _ENC.get(text)
    if r is None: r = _ENC[text] = tokenizer.encode(text, add_special_tokens=False).ids
    return r

MK = {"prev": enc(" <<previous>> "), "next": enc(" <<next>> "), "tgt": enc(" <<target>> ")}
SEP = enc(" | ")

def build_examples(docs, ctx, k_left=3, k_right=1, t_budget=96, l_budget=160, r_budget=64, max_sent=96):
    """One example per sentence. Context-aware input (for a causal LM the target comes LAST so it can attend to everything):
       <<previous>> left sentences <<next>> right sentences <<target>> target sentence."""
    exs = []
    for d in docs:
        toks = [enc(s)[:max_sent] for s in d["sents"]]
        n = len(toks)
        for i in range(n):
            tgt = toks[i][:t_budget]
            if ctx:
                left, right = [], []
                for j in range(max(0, i - k_left), i): left += toks[j] + SEP
                for j in range(i + 1, min(n, i + 1 + k_right)): right += toks[j] + SEP
                ids = MK["prev"] + left[-l_budget:] + MK["next"] + right[:r_budget] + MK["tgt"] + tgt
            else:
                ids = list(tgt)
            if not ids: ids = [0]
            rel = i / max(n - 1, 1)
            y = LABEL2ID[d["labels"][i]] if d["labels"] is not None and d["labels"][i] is not None else -1
            exs.append(dict(ids=ids, pos=[rel, rel * rel], y=y, doc=d["id"], i=i))
    return exs

def make_batches(exs, bs, shuffle, rng):
    idx = list(range(len(exs)))
    if shuffle: rng.shuffle(idx)
    batches = []
    for c in range(0, len(idx), bs * 50):
        ch = sorted(idx[c:c + bs * 50], key=lambda j: len(exs[j]["ids"]))
        batches += [ch[k:k + bs] for k in range(0, len(ch), bs)]
    if shuffle: rng.shuffle(batches)
    return batches

def collate(exs, bidx):
    import torch
    L = max(len(exs[j]["ids"]) for j in bidx)
    ids = torch.zeros((len(bidx), L), dtype=torch.long); mask = torch.zeros_like(ids)
    for r, j in enumerate(bidx):
        n = len(exs[j]["ids"]); ids[r, :n] = torch.tensor(exs[j]["ids"]); mask[r, :n] = 1
    pos = torch.tensor([exs[j]["pos"] for j in bidx], dtype=torch.float32)
    y = torch.tensor([exs[j]["y"] for j in bidx], dtype=torch.long)
    return ids, mask, pos, y


In [ ]:
# ================= CELL 3 — FIX TEST JSON PARSING =================

def parse_split(obj, style=None):
    """
    Supports:
      - Test: {document_id: [sentence1, sentence2, ...]}
      - Labeled documents: {document_id: {"text": [...], "labels": [...]}}
      - Lists of document dictionaries.
    """
    docs = []

    if isinstance(obj, dict):
        items = obj.items()
    elif isinstance(obj, list):
        items = enumerate(obj)
    else:
        raise ValueError(f"Unsupported split type: {type(obj).__name__}")

    for did, payload in items:
        # Test format: document ID -> list of sentence strings
        if isinstance(payload, list) and all(
            isinstance(sentence, str) for sentence in payload
        ):
            docs.append({
                "id": str(did),
                "sents": payload,
                "labels": None,
                "setter": None,
            })
            continue

        # Other existing formats: use the original extractor
        t, labels, setter, detected_style = _extract(payload, style)

        docs.append({
            "id": str(did),
            "sents": t,
            "labels": labels,
            "setter": setter,
        })

        if style is None and labels is not None:
            style = detected_style

    return docs, style


# Rebuild all domains and splits using the corrected parser
DOCS = {"CL": {}, "IT": {}}
TEST_OBJ = {}

for dom in ("CL", "IT"):
    for split in ("train", "dev"):
        DOCS[dom][split], _ = parse_split(RAW[f"{dom}_{split}.json"])

    TEST_OBJ[dom] = RAW[f"{dom}_test.json"]
    DOCS[dom]["test"], _ = parse_split(TEST_OBJ[dom])

    for split in ("train", "dev", "test"):
        print(f"{dom} {split}: {len(DOCS[dom][split])} documents")

    # Validate test sentence preservation
    for doc in DOCS[dom]["test"][:2]:
        assert doc["sents"], f"Empty test document: {doc['id']}"
        print(f"  {doc['id']}: {len(doc['sents'])} sentences")

print("DOCS rebuilt successfully.")

In [ ]:
# ================= CELL 7a — metrics + inference-time post-processing (numpy only) =================
def prf_report(y_true, y_pred, K):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    cm = np.zeros((K, K), dtype=np.int64); np.add.at(cm, (y_true, y_pred), 1)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    prec = np.divide(tp, tp + fp, out=np.zeros(K), where=(tp + fp) > 0)
    rec  = np.divide(tp, tp + fn, out=np.zeros(K), where=(tp + fn) > 0)
    f1   = np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)
    present = (cm.sum(0) + cm.sum(1)) > 0          # labels in gold or prediction = sklearn's default macro average
    return dict(macro_f1=float(f1[present].mean()), macro_p=float(prec[present].mean()), macro_r=float(rec[present].mean()),
                macro_f1_all=float(f1.mean()), acc=float(tp.sum() / cm.sum()), f1=f1, prec=prec, rec=rec,
                support=cm.sum(1), cm=cm, n_present=int(present.sum()))

def softmax_np(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def transition_logprobs(docs, label2id, K, alpha=1.0):
    T = np.full((K, K), alpha); S = np.full(K, alpha)
    for d in docs:
        ys = [label2id[l] for l in d["labels"]]
        S[ys[0]] += 1
        for a, b in zip(ys[:-1], ys[1:]): T[a, b] += 1
    return np.log(T / T.sum(1, keepdims=True)), np.log(S / S.sum())

def viterbi_smooth(logp, logT, logS, lam):
    """argmax_y  sum_i logp[i,y_i] + lam * (logS[y_0] + sum_i logT[y_{i-1}, y_i])"""
    n, K = logp.shape
    if lam == 0 or n == 1: return logp.argmax(1)
    dp = np.zeros((n, K)); bp = np.zeros((n, K), dtype=int)
    dp[0] = logp[0] + lam * logS
    for i in range(1, n):
        cand = dp[i - 1][:, None] + lam * logT
        bp[i] = cand.argmax(0); dp[i] = cand.max(0) + logp[i]
    path = [int(dp[-1].argmax())]
    for i in range(n - 1, 0, -1): path.append(int(bp[i, path[-1]]))
    return np.array(path[::-1])

def group_by_doc(exs):
    g = {}
    for j, e in enumerate(exs): g.setdefault(e["doc"], []).append(j)
    return {d: sorted(ix, key=lambda j: exs[j]["i"]) for d, ix in g.items()}

def smooth_split(probs, exs, logT, logS, lam):
    pred = np.zeros(len(exs), dtype=int)
    for d, ix in group_by_doc(exs).items():
        pred[ix] = viterbi_smooth(np.log(np.clip(probs[ix], 1e-9, 1)), logT, logS, lam)
    return pred

def knn_interpolate(q_emb, tr_emb, tr_y, q_probs, K, k=10, tau=0.1, lam=0.5):
    """p = lam * p_model + (1-lam) * p_kNN  (neighbourhood-at-inference idea; neighbours come from TRAIN sentences only)"""
    qe = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-9)
    te = tr_emb / (np.linalg.norm(tr_emb, axis=1, keepdims=True) + 1e-9)
    sims = qe @ te.T
    top = np.argpartition(-sims, kth=min(k, sims.shape[1] - 1), axis=1)[:, :k]
    out = np.zeros_like(q_probs)
    for r in range(len(qe)):
        s = sims[r, top[r]] / tau; w = softmax_np(s)
        np.add.at(out[r], tr_y[top[r]], w)
    return lam * q_probs + (1 - lam) * out

def validate_submission(obj, docs, label_set, style=None):
    """every document and sentence present, order preserved, exactly one valid label per sentence"""
    d2 = {d["id"]: d for d in docs}
    problems = []
    parsed, _ = parse_split(obj, style)
    if [d["id"] for d in parsed] != [d["id"] for d in docs]: problems.append("document ids / order differ from the test file")
    for p in parsed:
        ref = d2.get(p["id"])
        if ref is None: continue
        if p["sents"] != ref["sents"]: problems.append(f"{p['id']}: sentence text/order changed")
        if p["labels"] is None or len(p["labels"]) != len(ref["sents"]): problems.append(f"{p['id']}: label count mismatch"); continue
        bad = [l for l in p["labels"] if l not in label_set]
        if bad: problems.append(f"{p['id']}: invalid labels {set(bad)}")
    return problems


In [ ]:
# ================= CELL 6 — backbones, classifier, adapters (LoRA / rsLoRA / LoRMA+ / LoRMAπ / IA3 / QLoRA / prompt) =================
from transformers import GPT2Config, GPT2Model
from transformers.pytorch_utils import Conv1D

# ---- architecture registry: robust checkpoint discovery ----
# The previous filter only accepted paths containing "phase-1" or "phase1".
# If your Kaggle dataset folder uses a different name, that leaves ARCHS empty
# and next(iter(ARCHS)) raises StopIteration.
ARCHS = {}

def _is_gpt2_checkpoint(path):
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        if not isinstance(ck, dict):
            return False
        cfg = ck.get("model_config", {})
        sd = ck.get("model_state_dict", ck)
        keys = list(sd.keys()) if isinstance(sd, dict) else []
        return (
            isinstance(cfg, dict)
            and all(k in cfg for k in ("vocab_size", "context_length", "hidden_size", "num_layers", "num_heads"))
            and isinstance(sd, dict)
            and any(k.startswith("transformer.h.") or k.startswith("transformer.wte.") for k in keys)
        )
    except Exception:
        return False

# Prefer checkpoint_latest.pt; search all attached Kaggle inputs regardless of folder naming.
_candidates = (
    sorted(set(glob.glob("/kaggle/input/**/checkpoint_latest.pt", recursive=True)))
    + sorted(set(
        glob.glob("/kaggle/input/**/*.pt", recursive=True)
        + glob.glob("/kaggle/input/**/*.pth", recursive=True)
    ))
)
_candidates = list(dict.fromkeys(_candidates))
for _p in _candidates:
    if _is_gpt2_checkpoint(_p):
        ARCHS["gpt2_a"] = dict(path=_p, family="gpt2")
        break

# Optional manual override if auto-discovery does not find the intended checkpoint:
# ARCHS["gpt2_a"] = dict(path="/kaggle/input/YOUR_DATASET/checkpoint_latest.pt", family="gpt2")

print("registered architectures:", {k: v["path"] for k, v in ARCHS.items()})
if not ARCHS:
    print("WARNING: no compatible GPT-2-style Phase 1 checkpoint was auto-detected.")
    print("Run CELL 2 checkpoint inventory and set ARCHS manually to a compatible checkpoint.")
def build_backbone(arch, bb_dropout=0.1, n_layers=None):
    """-> (backbone, hidden, causal, n_positions, family). n_layers: keep only the first n layers (student initialisation)."""
    spec = ARCHS[arch]
    if spec["family"] != "gpt2":
        raise NotImplementedError(f"No loader for family '{spec['family']}'. Run the inventory cell, paste its output, and add a loader here.")
    ck = torch.load(spec["path"], map_location="cpu", weights_only=False); cfg = ck["model_config"]
    gcfg = GPT2Config(vocab_size=cfg["vocab_size"], n_positions=cfg["context_length"], n_ctx=cfg["context_length"], n_embd=cfg["hidden_size"],
                      n_layer=cfg["num_layers"], n_head=cfg["num_heads"], resid_pdrop=bb_dropout, embd_pdrop=bb_dropout, attn_pdrop=bb_dropout)
    bb = GPT2Model(gcfg)
    sd = ck["model_state_dict"]
    sd = {k.removeprefix("transformer."): v for k, v in sd.items() if k.startswith("transformer.")} or sd
    bb.load_state_dict(sd, strict=True); del ck
    if n_layers is not None and n_layers < len(bb.h):
        bb.h = nn.ModuleList(list(bb.h)[:n_layers]); bb.config.n_layer = n_layers
    return bb, cfg["hidden_size"], True, cfg["context_length"], "gpt2"

class SeqClassifier(nn.Module):
    def __init__(self, backbone, hidden, n_labels, causal=True, pos_dim=2, dropout=0.1, family="gpt2"):
        super().__init__()
        self.backbone, self.causal, self.pos_dim, self.family = backbone, causal, pos_dim, family
        self.drop = nn.Dropout(dropout); self.head = nn.Linear(hidden + pos_dim, n_labels); self.prompt = None
    def forward(self, ids, mask, pos=None, return_emb=False):
        B = ids.size(0); lengths = mask.sum(1)
        if self.prompt is not None:                       # prompt tuning: learnable vectors in front of the input
            emb = self.backbone.get_input_embeddings()(ids)
            P = self.prompt.shape[0]
            emb = torch.cat([self.prompt.unsqueeze(0).expand(B, -1, -1).to(emb.dtype), emb], 1)
            mask = torch.cat([torch.ones(B, P, device=ids.device, dtype=mask.dtype), mask], 1); lengths = lengths + P
            h = self.backbone(inputs_embeds=emb, attention_mask=mask, use_cache=False).last_hidden_state
        else:
            h = self.backbone(input_ids=ids, attention_mask=mask, use_cache=False).last_hidden_state
        if self.causal: pooled = h[torch.arange(B, device=h.device), lengths - 1]       # last real token sees the whole input
        else:
            m = mask.unsqueeze(-1).to(h.dtype); pooled = (h * m).sum(1) / m.sum(1)
        feats = pooled if (self.pos_dim == 0 or pos is None) else torch.cat([pooled, pos.to(pooled.dtype)], -1)
        logits = self.head(self.drop(feats))
        return (logits, pooled) if return_emb else logits

def count_params(m):
    tot = sum(p.numel() for p in m.parameters()); tr = sum(p.numel() for p in m.parameters() if p.requires_grad); return tot, tr
def model_size_mb(m):
    return (sum(p.numel() * p.element_size() for p in m.parameters()) + sum(b.numel() * b.element_size() for b in m.buffers())) / 2**20

# ---------------- adapter building blocks ----------------
class LoRADelta(nn.Module):                # additive:  dW x = (alpha/r) B A x   (rsLoRA: alpha/sqrt(r))
    def __init__(self, nx, nf, r, alpha, drop, rs=False):
        super().__init__()
        self.A = nn.Parameter(torch.empty(nx, r).uniform_(-1 / math.sqrt(nx), 1 / math.sqrt(nx))); self.B = nn.Parameter(torch.zeros(r, nf))
        self.scale = alpha / (math.sqrt(r) if rs else r); self.drop = nn.Dropout(drop)
    def forward(self, x): return (self.drop(x) @ self.A) @ self.B * self.scale

class LoRMAPlus(nn.Module):                # multiplicative, additive rank inflation: h' = (I + (alpha/r) B A) h ; B=0 at init
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d))); self.B = nn.Parameter(torch.zeros(d, r)); self.scale = alpha / r
    def forward(self, h): return h + ((h @ self.A.t()) @ self.B.t()) * self.scale      # = W0x + (alpha/r) B A (W0 x)

class LoRMAPi(nn.Module):                  # multiplicative, permutation-based rank inflation: h' = I_pi(BA) h, I_pi(BA) = I at init
    def __init__(self, d, r):
        super().__init__()
        B = torch.empty(d, r).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d)); B[:, 0] = 1.0
        A = torch.zeros(r, d); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A)
        i = torch.arange(d); self.register_buffer("idx", (i[None, :] - i[:, None]) % d)   # row i rotated right by i
    def matrix(self): return (self.B @ self.A).gather(1, self.idx)
    def forward(self, h): return h @ self.matrix().t().to(h.dtype)

class QVAdapt(nn.Module):                  # wraps GPT-2's fused c_attn (q|k|v); adapts q and v (k and v for IA3)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__()
        nx, nf = conv_dims(base); d = nf // 3; self.base, self.kind, self.d = base, kind, d
        if kind == "lora":         self.q, self.v = LoRADelta(nx, d, r, alpha, drop, rs), LoRADelta(nx, d, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.q, self.v = LoRMAPlus(d, r, alpha), LoRMAPlus(d, r, alpha)
        elif kind == "lorma_pi":   self.q, self.v = LoRMAPi(d, r), LoRMAPi(d, r)
        elif kind == "ia3":        self.lk, self.lv = nn.Parameter(torch.ones(d)), nn.Parameter(torch.ones(d))
        else: raise ValueError(kind)
    def forward(self, x):
        out = self.base(x); d = self.d; q, k, v = out[..., :d], out[..., d:2 * d], out[..., 2 * d:]
        if self.kind == "lora":            q, v = q + self.q(x), v + self.v(x)
        elif self.kind.startswith("lorma"): q, v = self.q(q), self.v(v)
        else:                              k, v = k * self.lk, v * self.lv
        return torch.cat([q, k, v], -1)

class IA3Input(nn.Module):                 # IA3 on the FFN: rescale the activation entering the down-projection
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(conv_dims(base)[0]))
    def forward(self, x): return self.base(x * self.l)

class LinAdapt(nn.Module):                 # same adapters for plain nn.Linear projections (non-GPT-2 backbones)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); self.base, self.kind = base, kind; di, do = base.in_features, base.out_features
        if kind == "lora": self.op = LoRADelta(di, do, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.op = LoRMAPlus(do, r, alpha)
        elif kind == "lorma_pi": self.op = LoRMAPi(do, r)
        elif kind == "ia3": self.l = nn.Parameter(torch.ones(do))
    def forward(self, x):
        y = self.base(x)
        if self.kind == "lora": return y + self.op(x)
        if self.kind == "ia3": return y * self.l
        return self.op(y)

def wrap_linear_targets(root, kind, names, **kw):
    n = 0
    for _, parent in list(root.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(parent, cname, LinAdapt(child, kind, **kw)); n += 1
    assert n > 0, f"no nn.Linear named {names} found in the backbone"

# ---------------- NF4 quantisation (QLoRA-style frozen base weights, 4-bit packed) ----------------
NF4_CODE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
    -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])
def nf4_quantize(w, block=64):
    flat = w.detach().float().reshape(-1); n = flat.numel(); pad = (-n) % block
    if pad: flat = F.pad(flat, (0, pad))
    blocks = flat.view(-1, block); absmax = blocks.abs().amax(dim=1).clamp_min(1e-8)
    normed = (blocks / absmax[:, None]).reshape(-1)
    code = NF4_CODE.to(flat.device); mids = (code[:-1] + code[1:]) / 2
    idx = torch.bucketize(normed, mids).to(torch.uint8)
    return ((idx[0::2] << 4) | idx[1::2]), absmax.half(), n
def nf4_dequant(packed, absmax, n, shape, block=64):
    code = NF4_CODE.to(packed.device)
    idx = torch.stack([(packed >> 4).long(), (packed & 0x0F).long()], dim=1).reshape(-1)
    return (code[idx].view(-1, block) * absmax.float()[:, None]).reshape(-1)[:n].view(shape)

class NF4Conv1D(nn.Module):
    def __init__(self, conv, block=64):
        super().__init__(); nx, nf = conv.weight.shape; self.nx, self.nf, self.block = nx, nf, block
        packed, absmax, n = nf4_quantize(conv.weight.data, block); self.n = n
        self.register_buffer("packed", packed); self.register_buffer("absmax", absmax); self.register_buffer("bias", conv.bias.data.clone())
    def forward(self, x):
        W = nf4_dequant(self.packed, self.absmax, self.n, (self.nx, self.nf), self.block).to(x.dtype)
        return x @ W + self.bias.to(x.dtype)

def conv_dims(m):
    if isinstance(m, NF4Conv1D): return m.nx, m.nf
    return tuple(m.weight.shape)                     # HF Conv1D weight is [in, out]

METHODS = ("full", "lora", "rslora", "qlora", "lorma_plus", "lorma_pi", "ia3", "prompt")
def apply_method(model, method, hp):
    assert method in METHODS, method
    if method == "full":
        for p in model.parameters(): p.requires_grad = True
        return model
    for p in model.backbone.parameters(): p.requires_grad = False
    r, alpha, drop = hp.get("r", 8), hp.get("alpha", 16), hp.get("adapter_dropout", 0.05)
    if method == "prompt":
        emb = model.backbone.get_input_embeddings().weight; P = hp.get("n_prompt", 20)
        init = emb[torch.randint(0, emb.shape[0], (P,))].detach().clone()
        model.prompt = nn.Parameter(init); return model
    kind = {"lora": "lora", "rslora": "lora", "qlora": "lora", "lorma_plus": "lorma_plus", "lorma_pi": "lorma_pi", "ia3": "ia3"}[method]
    kw = dict(r=r, alpha=alpha, drop=drop, rs=(method == "rslora"))
    if model.family == "gpt2":
        for blk in model.backbone.h:
            if method == "qlora":
                for owner, name in ((blk.attn, "c_attn"), (blk.attn, "c_proj"), (blk.mlp, "c_fc"), (blk.mlp, "c_proj")):
                    setattr(owner, name, NF4Conv1D(getattr(owner, name)))
            assert isinstance(blk.attn.c_attn, (Conv1D, NF4Conv1D)), "unexpected c_attn type in this transformers version"
            blk.attn.c_attn = QVAdapt(blk.attn.c_attn, kind, **kw)
            if kind == "ia3": blk.mlp.c_proj = IA3Input(blk.mlp.c_proj)
    else:
        if method == "qlora": raise NotImplementedError("QLoRA is implemented for the GPT-2 family only")
        names = ("k_proj", "v_proj") if kind == "ia3" else ("q_proj", "v_proj")
        wrap_linear_targets(model.backbone, kind, names, **({} if kind == "ia3" else kw))
    return model


In [ ]:
# ================= CELL 8 — training / evaluation harness + ledger =================
K = len(LABELS)
HP_DEFAULT = {
    "full": dict(lr=3e-5), "lora": dict(lr=5e-4, r=8, alpha=16), "rslora": dict(lr=5e-4, r=8, alpha=16), "qlora": dict(lr=5e-4, r=8, alpha=16),
    "lorma_plus": dict(lr=5e-4, r=8, alpha=16), "lorma_pi": dict(lr=5e-4, r=8), "ia3": dict(lr=3e-3), "prompt": dict(lr=1e-2, n_prompt=20),
}
CFG_DEFAULT = dict(arch=None, method="full", ctx=True, epochs=8, patience=3, bs=16, warmup=0.1, wd=0.01, head_lr=1e-3, label_smooth=0.05,
                   class_weights=True, bb_dropout=0.1, k_left=3, k_right=1, seed=SEED, n_layers=None, kd=None, merge_dev=False,
                   fixed_epochs=None, hp={}, force=False)
use_amp = device.type == "cuda"

@torch.no_grad()
def predict(model, exs, bs=64, emb=False):
    model.eval(); lg = np.zeros((len(exs), K), np.float32); em = None
    order = sorted(range(len(exs)), key=lambda j: len(exs[j]["ids"]))
    for s in range(0, len(order), bs):
        bidx = order[s:s + bs]; ids, mask, pos, _ = collate(exs, bidx)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            out = model(ids.to(device), mask.to(device), pos.to(device), return_emb=emb)
        if emb:
            out, pooled = out
            if em is None: em = np.zeros((len(exs), pooled.shape[-1]), np.float16)
            em[bidx] = pooled.float().cpu().numpy().astype(np.float16)
        lg[bidx] = out.float().cpu().numpy()
    return (lg, em) if emb else lg

def snapshot(model): return {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}
def restore(model, snap):
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in snap: p.copy_(snap[n].to(p.device))

def run_one(name, dom, cfg):
    c = {**CFG_DEFAULT, **cfg}; c["arch"] = c["arch"] or next(iter(ARCHS)); hp = {**HP_DEFAULT[c["method"]], **c["hp"]}
    rdir = WORK / "runs" / f"{name}_{dom}"; rdir.mkdir(parents=True, exist_ok=True)
    if (rdir / "result.json").exists() and not c["force"]:
        print(f"[{name}/{dom}] already done - skipping (force=True to redo)"); return json.load(open(rdir / "result.json"))
    seed_all(c["seed"]); t_all = time.time()
    tr_docs = DOCS[dom]["train"] + (DOCS[dom]["dev"] if c["merge_dev"] else [])
    kw = dict(ctx=c["ctx"], k_left=c["k_left"], k_right=c["k_right"])
    ex_tr, ex_dv, ex_te = build_examples(tr_docs, **kw), build_examples(DOCS[dom]["dev"], **kw), build_examples(DOCS[dom]["test"], **kw)
    y_tr = np.array([e["y"] for e in ex_tr]); y_dv = np.array([e["y"] for e in ex_dv])
    bb, hidden, causal, npos, fam = build_backbone(c["arch"], c["bb_dropout"], c["n_layers"])
    model = SeqClassifier(bb, hidden, K, causal, 2, 0.1, fam); apply_method(model, c["method"], hp); model.to(device)
    longest = max(len(e["ids"]) for e in ex_tr + ex_dv + ex_te) + (hp.get("n_prompt", 0) if c["method"] == "prompt" else 0)
    assert longest <= npos, f"input length {longest} exceeds the model context {npos}"
    tot_p, tr_p = count_params(model); size_mb = model_size_mb(model)
    cnt = np.bincount(y_tr, minlength=K).astype(float); cw = None
    if c["class_weights"]:
        w = 1 / np.sqrt(np.maximum(cnt, 1)); w = w * cnt.sum() / (w * cnt).sum(); cw = torch.tensor(w, dtype=torch.float32, device=device)
    T_logits = None
    if c["kd"]:
        assert not c["merge_dev"], "KD + merge_dev not supported"
        T_logits = np.load(WORK / "runs" / f"teacher_{dom}.npy"); assert len(T_logits) == len(ex_tr), "teacher logits do not match the training examples"
    other = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head.")]
    opt = torch.optim.AdamW([{"params": other, "lr": hp["lr"]}, {"params": list(model.head.parameters()), "lr": c["head_lr"]}], weight_decay=c["wd"])
    epochs = c["fixed_epochs"] or c["epochs"]; spe = math.ceil(len(ex_tr) / c["bs"]); total = spe * epochs; warm = int(c["warmup"] * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / max(1, warm), max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp); rng = random.Random(c["seed"])
    trainable = [p for p in model.parameters() if p.requires_grad]
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    print(f"[{name}/{dom}] arch={c['arch']} method={c['method']} ctx={c['ctx']} | train {len(ex_tr)} sents | params total {tot_p/1e6:.1f}M "
          f"trainable {tr_p/1e6:.3f}M ({100*tr_p/tot_p:.2f}%) | size {size_mb:.0f} MB")
    best, best_ep, bad, snap, hist = -1.0, 0, 0, None, []; t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); tl, nb = 0.0, 0
        for bidx in make_batches(ex_tr, c["bs"], True, rng):
            ids, mask, pos, y = collate(ex_tr, bidx); ids, mask, pos, y = ids.to(device), mask.to(device), pos.to(device), y.to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp): logits = model(ids, mask, pos)
            loss = F.cross_entropy(logits.float(), y, weight=cw, label_smoothing=c["label_smooth"])
            if T_logits is not None:
                T, a = c["kd"].get("T", 2.0), c["kd"].get("alpha", 0.5); tl_ = torch.tensor(T_logits[bidx], device=device)
                kl = F.kl_div(F.log_softmax(logits.float() / T, -1), F.softmax(tl_ / T, -1), reduction="batchmean") * T * T
                loss = a * loss + (1 - a) * kl
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0); scaler.step(opt); scaler.update(); sched.step(); tl += loss.item(); nb += 1
        if c["merge_dev"]:
            print(f"   epoch {ep}/{epochs} loss {tl/nb:.4f} (train+dev refit: no validation)"); continue
        rep = prf_report(y_dv, predict(model, ex_dv).argmax(1), K); hist.append(dict(epoch=ep, loss=tl / nb, dev_macro_f1=rep["macro_f1"]))
        print(f"   epoch {ep}/{epochs} loss {tl/nb:.4f} | dev macro-F1 {rep['macro_f1']:.4f} (P {rep['macro_p']:.3f} R {rep['macro_r']:.3f}) acc {rep['acc']:.3f}")
        if rep["macro_f1"] > best + 1e-4: best, best_ep, bad, snap = rep["macro_f1"], ep, 0, snapshot(model)
        else:
            bad += 1
            if bad >= c["patience"] and not c["fixed_epochs"]: print("   early stop"); break
    train_sec = time.time() - t0
    if snap is not None: restore(model, snap)
    peak_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else 0.0
    t1 = time.time(); lg_dv, em_dv = predict(model, ex_dv, emb=True); sps = len(ex_dv) / max(time.time() - t1, 1e-6)
    lg_te, em_te = predict(model, ex_te, emb=True); lg_tr, em_tr = predict(model, ex_tr, emb=True)
    for nm, arr in dict(dev_logits=lg_dv, test_logits=lg_te, train_logits=lg_tr, dev_emb=em_dv, test_emb=em_te, train_emb=em_tr).items(): np.save(rdir / f"{nm}.npy", arr)
    np.save(rdir / "train_y.npy", y_tr)
    rep = prf_report(y_dv, lg_dv.argmax(1), K)
    res = dict(name=name, domain=dom, arch=c["arch"], method=c["method"], ctx=c["ctx"], kd=bool(c["kd"]), merge_dev=c["merge_dev"], n_layers=c["n_layers"],
               best_epoch=best_ep or epochs, epochs_run=len(hist) or epochs, dev_macro_f1=rep["macro_f1"], dev_macro_p=rep["macro_p"], dev_macro_r=rep["macro_r"],
               dev_macro_f1_all13=rep["macro_f1_all"], dev_acc=rep["acc"], params_total=tot_p, params_trainable=tr_p, trainable_pct=100 * tr_p / tot_p,
               model_size_mb=size_mb, peak_gpu_mb=peak_mb, train_sec=train_sec, sents_per_sec=sps, cfg=json.dumps({**c, "hp": hp}, default=str),
               **{f"f1_{l}": float(rep["f1"][i]) for i, l in enumerate(LABELS)})
    json.dump({**res, "confusion": rep["cm"].tolist(), "history": hist}, open(rdir / "result.json", "w"))
    pd.DataFrame([res]).to_csv(LEDGER, mode="a", header=not LEDGER.exists(), index=False)
    print(f"[{name}/{dom}] DONE dev macro-F1 {rep['macro_f1']:.4f} @epoch {res['best_epoch']} | {train_sec/60:.1f} min | peak {peak_mb:.0f} MB | {sps:.0f} sent/s")
    del model, opt, scaler, snap; gc.collect(); torch.cuda.empty_cache()
    return res

def run_experiment(name, domains=("CL", "IT"), **cfg): return {d: run_one(name, d, cfg) for d in domains}

def make_teacher(run_names, dom):
    """geometric-mean ensemble of finished runs' logits on the TRAIN sentences -> teacher for knowledge distillation"""
    lp = [np.log(softmax_np(np.load(WORK / "runs" / f"{n}_{dom}" / "train_logits.npy")) + 1e-9) for n in run_names]
    np.save(WORK / "runs" / f"teacher_{dom}.npy", np.mean(lp, 0).astype(np.float32)); print(f"teacher_{dom}: ensemble of {run_names}")

def show_ledger(cols=None):
    df = pd.read_csv(LEDGER).drop_duplicates(subset=["name", "domain"], keep="last")
    cols = cols or ["name", "domain", "arch", "method", "ctx", "kd", "dev_macro_f1", "dev_macro_p", "dev_macro_r", "best_epoch", "params_trainable", "model_size_mb", "peak_gpu_mb", "train_sec"]
    print(df[cols].round(4).to_string(index=False)); return df


In [ ]:
# ================= CELL 9 — smoke tests (run once BEFORE any long training; ~2-3 min) =================
def smoke_tests(arch=None):
    if not ARCHS:
        raise RuntimeError(
            "ARCHS is empty: no compatible Phase 1 checkpoint was detected. "
            "Run CELL 2 to inspect checkpoints, then edit ARCHS in CELL 6 "
            "with the correct checkpoint path and family='gpt2' for the GPT-2 checkpoint."
        )
    arch = arch or next(iter(ARCHS))
    if arch not in ARCHS:
        raise KeyError(f"Unknown architecture {arch!r}. Registered: {list(ARCHS)}")
    print("testing architecture:", arch)
    bb, hidden, causal, npos, fam = build_backbone(arch, bb_dropout=0.0)
    base = SeqClassifier(bb, hidden, K, causal, 2, 0.0, fam).eval()
    exs = build_examples(DOCS["CL"]["train"][:1], True)[:4]; ids, mask, pos, y = collate(exs, list(range(len(exs))))
    with torch.no_grad(): ref = base(ids, mask, pos)
    rows = []
    for m in ("lora", "rslora", "lorma_plus", "lorma_pi", "ia3", "qlora", "prompt"):
        mdl = copy.deepcopy(base); apply_method(mdl, m, {"n_prompt": 8}); mdl.eval()
        with torch.no_grad(): out = mdl(ids, mask, pos)
        diff = (out - ref).abs().max().item(); tot, tr = count_params(mdl)
        if m in ("qlora", "prompt"): ok = bool(torch.isfinite(out).all()); note = "changes outputs by design (quantisation / extra tokens)"
        else: ok = diff < 1e-3; note = "must equal the base model at initialisation"
        rows.append(dict(method=m, max_logit_diff=round(diff, 5), trainable_M=round(tr / 1e6, 3), size_MB=round(model_size_mb(mdl)), ok=ok, check=note))
        del mdl; gc.collect()
    pi = apply_method(copy.deepcopy(base), "lorma_pi", {}).backbone.h[0].attn.c_attn.q.matrix()
    rows.append(dict(method="lorma_pi: matrix==I", max_logit_diff=round((pi - torch.eye(pi.shape[0])).abs().max().item(), 6), ok=bool(torch.allclose(pi, torch.eye(pi.shape[0]), atol=1e-6)), check="permutation inflation starts as identity"))
    df = pd.DataFrame(rows); print(df.to_string(index=False)); assert df["ok"].all(), "an adapter test failed - do not run that method"
    if device.type == "cuda":
        for m in ("lorma_plus", "full"):
            mdl = SeqClassifier(*build_backbone(arch, 0.0)[:2], K, causal, 2, 0.0, fam); apply_method(mdl, m, {}); mdl.to(device).train()
            ids_, mask_, pos_, y_ = [t.to(device) for t in collate(exs, list(range(len(exs))))]
            opt = torch.optim.AdamW([p for p in mdl.parameters() if p.requires_grad], lr=1e-3 if m != "full" else 1e-4); L = []
            for _ in range(25):
                loss = F.cross_entropy(mdl(ids_, mask_, pos_), y_); opt.zero_grad(); loss.backward(); opt.step(); L.append(loss.item())
            print(f"overfit test [{m}]: loss {L[0]:.3f} -> {L[-1]:.3f}"); assert L[-1] < 0.7 * L[0], f"{m} did not learn"
            del mdl, opt; gc.collect(); torch.cuda.empty_cache()
    print("\nALL SMOKE TESTS PASSED")
smoke_tests()


### Smoke tests passed? Then run the experiments in this order (compulsory ones first). Each run skips itself if its result already exists, so it is safe to re-run after a crash.

In [ ]:
# ================= CELL 10 — experiment plan (edit ONLY the `arch` of each entry once the four Phase 1 models are registered) =================
if not ARCHS:
    raise RuntimeError("ARCHS is empty. Fix checkpoint registration in CELL 6 before defining experiments.")
ARCH_DEFAULT = next(iter(ARCHS))
PLAN = {   # compulsory ones first: LoRMA and KD must exist even if time runs short
    "E0b_full_ctx":  dict(method="full",       ctx=True,  arch=ARCH_DEFAULT),   # baseline: full fine-tuning, context-aware
    "E2_lorma_plus": dict(method="lorma_plus", ctx=True,  arch=ARCH_DEFAULT),   # COMPULSORY: low-rank multiplicative adaptation (LoRMA+)
    "E1_lora":       dict(method="lora",       ctx=True,  arch=ARCH_DEFAULT),
    "E3_ia3":        dict(method="ia3",        ctx=True,  arch=ARCH_DEFAULT),
    "E4_qlora":      dict(method="qlora",      ctx=True,  arch=ARCH_DEFAULT),   # NF4-quantised frozen backbone + LoRA
    "E5_prompt":     dict(method="prompt",     ctx=True,  arch=ARCH_DEFAULT),   # prompt tuning (optional 5th/6th technique)
    "E0a_full_sent": dict(method="full",       ctx=False, arch=ARCH_DEFAULT),   # context ablation: sentence-only
}
print(pd.DataFrame(PLAN).T.to_string())


In [ ]:
# ================= CELL 11 — RUN A: baseline + the compulsory LoRMA experiment (both domains) =================
for nm in ("E0b_full_ctx", "E2_lorma_plus"):
    run_experiment(nm, **PLAN[nm])
show_ledger();


In [ ]:
# ================= CELL 12 — RUN B: the other adaptation techniques + context ablation =================
for nm in ("E1_lora", "E3_ia3", "E4_qlora", "E0a_full_sent"):     # add "E5_prompt" if you want a sixth technique
    run_experiment(nm, **PLAN[nm])
show_ledger();


In [ ]:
# ================= CELL 13 — RUN C: knowledge distillation (teacher = ensemble of finished runs, student = half-depth model from Phase 1 weights) =================
TEACHERS = ["E0b_full_ctx", "E2_lorma_plus"]          # choose the strongest finished runs
for dom in ("CL", "IT"): make_teacher(TEACHERS, dom)
# Student initialised from the same Phase 1 weights but keeping only the first 6 layers (set n_layers=None and change `arch` to use a smaller Phase 1 model instead).
run_experiment("E6_kd_student", method="full", ctx=True, arch=ARCH_DEFAULT, n_layers=6, kd=dict(T=2.0, alpha=0.5))
# Control without distillation (same student, hard labels only) so the benefit of KD is measurable:
run_experiment("E6_student_nokd", method="full", ctx=True, arch=ARCH_DEFAULT, n_layers=6)
show_ledger();


In [ ]:
# ================= CELL 14 — comparison tables, per-class F1, confusion matrices, architecture coverage =================
import matplotlib; import matplotlib.pyplot as plt
df = pd.read_csv(LEDGER).drop_duplicates(subset=["name", "domain"], keep="last")
df = df[~df["merge_dev"].astype(bool)]
main = df.pivot_table(index="name", columns="domain", values="dev_macro_f1").round(4); main["mean"] = main.mean(1)
print("dev macro-F1 by experiment and domain\n", main.sort_values("mean", ascending=False).to_string())
eff = df.groupby("name")[["params_trainable", "model_size_mb", "peak_gpu_mb", "train_sec", "sents_per_sec"]].mean().round(1); print("\nefficiency (mean over domains)\n", eff.to_string())
pc = df.set_index(["name", "domain"])[[f"f1_{l}" for l in LABELS]].round(3); pc.columns = LABELS; print("\nper-class F1\n", pc.to_string())
print("\narchitecture coverage (all four Phase 1 architectures must appear):\n", df.groupby("arch")["name"].nunique().to_string())
def plot_cm(name, dom):
    r = json.load(open(WORK / "runs" / f"{name}_{dom}" / "result.json")); cm = np.array(r["confusion"]); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(8, 7)); im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(K)); ax.set_xticklabels(LABELS, rotation=90, fontsize=7); ax.set_yticks(range(K)); ax.set_yticklabels(LABELS, fontsize=7)
    ax.set_title(f"{name} / {dom}  (row-normalised)"); plt.colorbar(im); plt.tight_layout(); plt.savefig(WORK / f"cm_{name}_{dom}.png", dpi=120); plt.show()
BEST = main["mean"].idxmax(); print("\nbest by mean dev macro-F1:", BEST)
for dom in ("CL", "IT"): plot_cm(BEST, dom)


In [ ]:
# ================= CELL 15 — post-hoc improvements with NO retraining: ensembling, transition smoothing, kNN interpolation =================
def load_arr(name, dom, nm): return np.load(WORK / "runs" / f"{name}_{dom}" / f"{nm}.npy")
def ens_probs(names, dom, split):
    lp = [np.log(softmax_np(load_arr(n, dom, f"{split}_logits")) + 1e-9) for n in names]; return softmax_np(np.mean(lp, 0))
def posthoc_probs(names, dom, split, smooth=0.0, knn_lam=1.0, k=10, tau=0.1):
    p = ens_probs(names, dom, split)
    if knn_lam < 1.0:   # neighbours from train embeddings of the FIRST run in `names`
        p = knn_interpolate(load_arr(names[0], dom, f"{split}_emb").astype(np.float32), load_arr(names[0], dom, "train_emb").astype(np.float32),
                            load_arr(names[0], dom, "train_y"), p, K, k=k, tau=tau, lam=knn_lam)
    return p
def predict_labels(probs, exs, logT, logS, smooth): return smooth_split(probs, exs, logT, logS, smooth) if smooth > 0 else probs.argmax(1)

def tune_posthoc(names, dom):
    ex_dv = build_examples(DOCS[dom]["dev"], True); y = np.array([e["y"] for e in ex_dv])
    logT, logS = transition_logprobs(DOCS[dom]["train"], LABEL2ID, K); rows = []
    for knn_lam in (1.0, 0.8, 0.6):
        p = posthoc_probs(names, dom, "dev", knn_lam=knn_lam)
        for sm in (0.0, 0.25, 0.5, 1.0):
            rows.append(dict(domain=dom, knn_lam=knn_lam, smooth=sm, dev_macro_f1=prf_report(y, predict_labels(p, ex_dv, logT, logS, sm), K)["macro_f1"]))
    t = pd.DataFrame(rows); base = t[(t.knn_lam == 1.0) & (t.smooth == 0.0)].dev_macro_f1.iloc[0]
    print(t.round(4).to_string(index=False)); print(f"baseline {base:.4f}; best {t.dev_macro_f1.max():.4f}"); return t, base

CANDIDATES = {dom: [BEST] for dom in ("CL", "IT")}      # list one or more finished run names per domain (an ensemble if several)
for dom in ("CL", "IT"): _ = tune_posthoc(CANDIDATES[dom], dom)
print("\nRule: adopt a post-hoc setting only if it gains >= 0.01 on BOTH domains; dev has very few documents and small gains are noise.")


In [ ]:
# ================= CELL 16 — FINAL PREDICTIONS + VALIDATION + ZIP =================
# The observed test format is {document_id: [sentence, ...]}.
# Build a separate predictions object: {document_id: [predicted_label, ...]}.
FINAL = {
    "CL": dict(names=[BEST], knn_lam=1.0, smooth=0.0),
    "IT": dict(names=[BEST], knn_lam=1.0, smooth=0.0),
}

PREDICTIONS = {}
for dom in ("CL", "IT"):
    f = FINAL[dom]
    ex_te = build_examples(DOCS[dom]["test"], ctx=PLAN[f["names"][0]]["ctx"])
    probs = posthoc_probs(f["names"], dom, "test", knn_lam=f["knn_lam"])
    logT, logS = transition_logprobs(
        DOCS[dom]["train"] + DOCS[dom]["dev"], LABEL2ID, K
    )
    yhat = predict_labels(probs, ex_te, logT, logS, f["smooth"])

    by_doc = defaultdict(list)
    for e, y_ in sorted(zip(ex_te, yhat), key=lambda pair: (pair[0]["doc"], pair[0]["i"])):
        by_doc[e["doc"]].append(LABELS[int(y_)])

    pred_obj = {}
    for d in DOCS[dom]["test"]:
        labs = by_doc[d["id"]]
        assert len(labs) == len(d["sents"]), (
            f"{dom}/{d['id']}: predicted {len(labs)} labels for {len(d['sents'])} sentences"
        )
        assert all(lab in LABEL2ID for lab in labs)
        pred_obj[d["id"]] = labs

    # Validate exact IDs/order and sentence-level label counts against the input test set.
    assert list(pred_obj.keys()) == list(TEST_OBJ[dom].keys()), f"{dom}: document IDs/order mismatch"
    for doc_id, sentences in TEST_OBJ[dom].items():
        assert len(pred_obj[doc_id]) == len(sentences), f"{dom}/{doc_id}: label count mismatch"

    PREDICTIONS[dom] = pred_obj
    out_file = WORK / f"prediction_{dom}.json"
    with open(out_file, "w", encoding="utf-8") as f_out:
        json.dump(pred_obj, f_out, ensure_ascii=False, indent=2)

    print(f"{dom}: {sum(map(len, pred_obj.values()))} sentence labels")
    print("  label distribution:", dict(Counter(l for labs in pred_obj.values() for l in labs)))
    print("  saved:", out_file)

# Combined keyed output; check the capstone submission instructions for the required final filename/schema.
combined = {"CL": PREDICTIONS["CL"], "IT": PREDICTIONS["IT"]}
combined_file = WORK / "prediction.json"
with open(combined_file, "w", encoding="utf-8") as f_out:
    json.dump(combined, f_out, ensure_ascii=False, indent=2)

zip_path = Path("/kaggle/working/submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(combined_file, "prediction.json")
    zf.write(WORK / "prediction_CL.json", "prediction_CL.json")
    zf.write(WORK / "prediction_IT.json", "prediction_IT.json")

with zipfile.ZipFile(zip_path) as zf:
    print("ZIP files:", zf.namelist())
    check = json.load(zf.open("prediction.json"))
    assert list(check.keys()) == ["CL", "IT"]
    for dom in ("CL", "IT"):
        assert list(check[dom].keys()) == list(TEST_OBJ[dom].keys())
        for doc_id, sents in TEST_OBJ[dom].items():
            assert len(check[dom][doc_id]) == len(sents)

print("Prediction files written and structural validation passed.")
